In [3]:
import yfinance as yf
import numpy as np
import matplotlib.pyplot as plt
import torch
from transformers import GPT2LMHeadModel, GPT2Tokenizer

In [5]:
ticker = "^NSEI"
start_date = "2023-01-01"
end_date = "2023-06-08"
data = yf.download(ticker, start=start_date, end=end_date)

prices = data["Close"]

[*********************100%***********************]  1 of 1 completed


In [10]:
prices = prices.values.tolist()

In [11]:
print(prices)

[[18197.44921875], [18232.55078125], [18042.94921875], [17992.150390625], [17859.44921875], [18101.19921875], [17914.150390625], [17895.69921875], [17858.19921875], [17956.599609375], [17894.849609375], [18053.30078125], [18165.349609375], [18107.849609375], [18027.650390625], [18118.55078125], [18118.30078125], [17891.94921875], [17604.349609375], [17648.94921875], [17662.150390625], [17616.30078125], [17610.400390625], [17854.05078125], [17764.599609375], [17721.5], [17871.69921875], [17893.44921875], [17856.5], [17770.900390625], [17929.849609375], [18015.849609375], [18035.849609375], [17944.19921875], [17844.599609375], [17826.69921875], [17554.30078125], [17511.25], [17465.80078125], [17392.69921875], [17303.94921875], [17450.900390625], [17321.900390625], [17594.349609375], [17711.44921875], [17754.400390625], [17589.599609375], [17412.900390625], [17154.30078125], [17043.30078125], [16972.150390625], [16985.599609375], [17100.05078125], [16988.400390625], [17107.5], [17151.9003

In [ ]:
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
model = GPT2LMHeadModel.from_pretrained("gpt2")

In [ ]:
encoded_prices = tokenizer.encode(" ".join([str(price) for price in prices]), return_tensors="pt")

model.train()
optimizer = torch.optim.Adam(model.parameters(), lr=5e-5)
model.resize_token_embeddings(len(tokenizer))

for _ in range(3):
    model.zero_grad()
    outputs = model(encoded_prices, labels=encoded_prices)
    loss = outputs.loss
    loss.backward()
    optimizer.step()

In [ ]:
generated = model.generate(encoded_prices, max_length=len(encoded_prices) + 10, temperature=1.0, num_return_sequences=1)
generated_prices = tokenizer.decode(generated[0], skip_special_tokens=True).split()

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(data.index, prices, label="Historical Prices")
plt.plot(data.index[-1] + pd.to_timedelta(np.arange(1, len(generated_prices) + 1), 'D'), [float(price) for price in generated_prices[len(prices):]], "g^", label="Predicted Prices")
plt.xlabel("Date")
plt.ylabel("Stock Price")
plt.title(f"{ticker} - Historical and Predicted Stock Prices (GPT)")
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()